# Week 18: Autonomous AI Agents & Multi-Agent Systems
## Notebook 08: State Machine Agent Architectures: Cyclic Graphs & Reducers

### 1. Conceptual Intuition & Theoretical Background
Early agent frameworks (like original LangChain chains) modeled execution as Directed Acyclic Graphs (DAGs) or linear chains. However, real-world autonomous problem solving is **inherently cyclic**:
$$\text{Plan} \longrightarrow \text{Execute Tool} \longrightarrow \text{Inspect Result} \longrightarrow \text{Evaluate Failure} \longrightarrow \text{Loop back to Plan}$$

**LangGraph / State Machine Agent Architecture** formalizes agents as **Cyclic Graphs**:
1. **Agent State ($S_t$):** A centralized typed schema representing current context (message history, working variables, tool calls).
2. **State Reducers ($R$):** Deterministic functions that update state:
$$S_{t+1} = R(S_t, \Delta S)$$
3. **Conditional Edge Routers:** Functions that inspect $S_t$ and dynamically route execution to the next node (e.g. `should_continue()` deciding between `execute_tool` or `end`).
4. **Human-in-the-Loop Checkpoints:** Persisting state to disk at designated nodes, allowing human review before authorizing sensitive tool actions.

---

### 2. Scratch Python Implementation: Cyclic Agent Graph Engine

In [ ]:
import json

class CyclicAgentGraph:
    def __init__(self):
        self.nodes = {}
        self.conditional_edges = {}
        self.checkpoints = []
        
    def add_node(self, name, func):
        self.nodes[name] = func
        
    def add_conditional_edge(self, source_node, routing_func):
        self.conditional_edges[source_node] = routing_func
        
    def run(self, initial_state, max_cycles=10):
        current_state = initial_state.copy()
        current_node = 'planner'
        cycles = 0
        
        print(f"Starting Cyclic Graph Execution from: [{current_node}]")
        
        while current_node != 'END' and cycles < max_cycles:
            cycles += 1
            # Checkpoint state
            self.checkpoints.append((current_node, current_state.copy()))
            
            # Execute node logic
            node_fn = self.nodes[current_node]
            state_delta = node_fn(current_state)
            
            # Reducer: update state
            current_state.update(state_delta)
            
            # Route to next node
            if current_node in self.conditional_edges:
                next_node = self.conditional_edges[current_node](current_state)
            else:
                next_node = 'END'
                
            print(f"Cycle {cycles}: Executed [{current_node}] -> Routed to -> [{next_node}]")
            current_node = next_node
            
        return current_state

# Define Node Handlers
def planner_node(state):
    attempts = state.get('attempts', 0)
    print(f"  [Planner] Analyzing problem. Current attempt count: {attempts}")
    return {'attempts': attempts + 1}

def tool_execution_node(state):
    print("  [Tool Executor] Running simulated database query...")
    # Simulate success only on attempt 2
    if state['attempts'] >= 2:
        return {'status': 'SUCCESS', 'data': 'Target DB record retrieved!'}
    else:
        return {'status': 'ERROR', 'error_msg': 'Connection timeout.'}

def router_after_tool(state):
    if state.get('status') == 'SUCCESS':
        return 'END'
    else:
        print("  [Router] Tool failed! Looping back to Planner to retry...")
        return 'planner' # CYCLIC LOOP BACK!

# Assemble Graph
agent_graph = CyclicAgentGraph()
agent_graph.add_node('planner', planner_node)
agent_graph.add_node('executor', tool_execution_node)

agent_graph.add_conditional_edge('planner', lambda s: 'executor')
agent_graph.add_conditional_edge('executor', router_after_tool)

# Run Graph
initial_state = {'query': 'Find active subscriptions', 'attempts': 0}
final_state = agent_graph.run(initial_state)

print("\n--- Final Execution State ---")
print(json.dumps(final_state, indent=2))

### 3. Step-by-Step Code Walkthrough
1. **Cyclic Routing**: When the tool execution fails, the router returns `'planner'`, forming a cyclic execution loop.
2. **State Reducer**: Node return values are merged into the persistent state dictionary.
3. **Execution Checkpointing**: States are saved at each step for debugging and rollback.

---

### 4. Industry Applications & Trade-offs
- **Coding Agents (SWE-bench):** Coding loop: write code -> run pytest -> parse trace -> loop back to edit file until tests pass.
- **Trade-off:** Unconstrained cyclic graphs can enter infinite loops without explicit max-cycle bounds.

---

### 5. Top Interview Questions & Answers
**Q1: Why are Directed Acyclic Graphs (DAGs) insufficient for autonomous AI agents?**
> **Answer:** DAGs only support forward execution. Autonomous agents require cycles and loops to recover from tool errors, reflect on incorrect outputs, and retry actions based on execution feedback.

**Q2: What is the purpose of State Reducers in multi-agent graph architectures?**
> **Answer:** State reducers define deterministic rules for merging incremental updates into the global state, preventing race conditions when multiple agent nodes execute in parallel.